In [29]:
import pandas as pd
from pathlib import Path
import numpy as np
import os

#1. Define the path and load the necessary data for the readmission task ---
# We'll start with Hospital A
DATA_PATH = Path("data") / "/Users/faezehhosseini/phd_federated_learning_project/data/hospital_A/csv"
patients_df = pd.read_csv(DATA_PATH / "patients.csv")
encounters_df = pd.read_csv(DATA_PATH / "encounters.csv")

#2. Data Cleaning and Preparation ---

# Convert date columns to datetime objects, removing timezone info
patients_df['BIRTHDATE'] = pd.to_datetime(patients_df['BIRTHDATE'])
encounters_df['START'] = pd.to_datetime(encounters_df['START']).dt.tz_localize(None)
encounters_df['STOP'] = pd.to_datetime(encounters_df['STOP']).dt.tz_localize(None)

#3. Identify Hospital Admissions (Inpatient Encounters) ---
# Readmission is only relevant for patients who were formally admitted.
inpatient_df = encounters_df[encounters_df['ENCOUNTERCLASS'] == 'inpatient'].copy()
print(f"Found {len(inpatient_df)} total inpatient encounters.")

# Sort encounters by patient and start date to create a timeline for each patient
inpatient_df = inpatient_df.sort_values(by=['PATIENT', 'START'], ascending=True)

#4. Engineer the Target Variable: `readmitted_30d` ---

# For each patient, calculate the date of their next admission
# We group by patient and shift the 'START' date up by one row.
inpatient_df['NEXT_ADMISSION_DATE'] = inpatient_df.groupby('PATIENT')['START'].shift(-1)

# Calculate the time (in days) between a discharge and the next admission
inpatient_df['DAYS_TO_NEXT_ADMISSION'] = (inpatient_df['NEXT_ADMISSION_DATE'] - inpatient_df['STOP']).dt.days

# Create the target variable: 1 if readmitted within 30 days, 0 otherwise.
# We check if the time to next admission is between 0 and 30 days.
inpatient_df['readmitted_30d'] = np.where(
    (inpatient_df['DAYS_TO_NEXT_ADMISSION'] >= 0) & (inpatient_df['DAYS_TO_NEXT_ADMISSION'] <= 30),
    1,
    0
)

#5. Create the Base Feature Set ---
# Merge with patient demographics
model_df = pd.merge(
    inpatient_df,
    patients_df[['Id', 'BIRTHDATE', 'GENDER', 'RACE', 'ETHNICITY', 'ZIP']],
    left_on='PATIENT',
    right_on='Id',
    how='left'
)

# Calculate patient's age at the time of admission
model_df['PATIENT_AGE'] = (model_df['START'] - model_df['BIRTHDATE']).dt.days / 365.25
model_df['PATIENT_AGE'] = model_df['PATIENT_AGE'].apply(lambda x: max(0, x)).astype(int)


#6. Final Data Selection and Inspection ---
# Select the columns we need for our initial model
final_df = model_df[[
    'PATIENT',
    'START',
    'STOP',
    'PATIENT_AGE',
    'GENDER',
    'RACE',
    'ETHNICITY',
    'ZIP',
    'readmitted_30d' # This is our target
]].copy()


print("\n--- Preprocessed Data for Readmission Task ---")
print("Shape of the final DataFrame:", final_df.shape)
print("\nReadmission counts:")
print(final_df['readmitted_30d'].value_counts())

print("\nExample of a readmitted patient encounter (readmitted_30d = 1):")
# Find an original encounter that led to a readmission to show the logic
readmission_examples = model_df[model_df['readmitted_30d'] == 1]
display(readmission_examples[['PATIENT', 'STOP', 'NEXT_ADMISSION_DATE', 'DAYS_TO_NEXT_ADMISSION', 'readmitted_30d']].head())

print("\nFinal model-ready DataFrame head:")
display(final_df.head())

Found 1022 total inpatient encounters.

--- Preprocessed Data for Readmission Task ---
Shape of the final DataFrame: (1022, 9)

Readmission counts:
readmitted_30d
0    828
1    194
Name: count, dtype: int64

Example of a readmitted patient encounter (readmitted_30d = 1):


,PATIENT,STOP,NEXT_ADMISSION_DATE,DAYS_TO_NEXT_ADMISSION,readmitted_30d
1,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2007-08-31 03:00:22,2007-09-28 05:00:22,28.0,1
5,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2008-02-22 00:56:02,2008-03-23 13:56:02,30.0,1
6,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2008-03-28 21:11:02,2008-04-27 07:11:02,29.0,1
9,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2008-09-05 07:04:02,2008-10-04 13:04:02,29.0,1
14,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2009-06-16 23:31:02,2009-07-17 06:31:02,30.0,1



Final model-ready DataFrame head:


,PATIENT,START,STOP,PATIENT_AGE,GENDER,RACE,ETHNICITY,ZIP,readmitted_30d
0,012ba41c-1649-1326-8acf-0d3f0aa40b2c,1963-12-25 18:43:54,1963-12-26 18:43:54,36,M,white,hispanic,2134,0
1,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2007-08-26 19:51:22,2007-08-31 03:00:22,79,M,white,hispanic,2134,1
2,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2007-09-28 05:00:22,2007-10-03 08:52:22,79,M,white,hispanic,2134,0
3,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2007-11-16 17:32:02,2007-11-21 02:08:02,79,M,white,hispanic,2134,0
4,012ba41c-1649-1326-8acf-0d3f0aa40b2c,2007-12-25 13:08:02,2008-01-07 15:45:02,80,M,white,hispanic,2134,0


In [30]:
import pandas as pd
from pathlib import Path
import numpy as np

def build_readmission_cohort(data_path: Path, top_k_conditions=50):
    """
    Builds a model-ready cohort for a 30-day readmission task from raw Synthea CSV files.
    
    This function incorporates advanced feature engineering, including utilization history
    and a bag-of-words representation for prior medical conditions.
    
    Args:
        data_path (Path): The path to the directory containing the Synthea CSV files.
        top_k_conditions (int): The number of most common conditions to use as features.

    Returns:
        pd.DataFrame: A DataFrame containing the features (X).
        pd.Series: A Series containing the target variable (y).
    """
    
    # --- 1. Load all necessary data ---
    patients_df = pd.read_csv(data_path / "patients.csv")
    encounters_df = pd.read_csv(data_path / "encounters.csv")
    conditions_df = pd.read_csv(data_path / "conditions.csv")
    meds_df = pd.read_csv(data_path / "medications.csv")

    # --- 2. Data Cleaning and Preparation ---
    for df, cols in [(patients_df, ['BIRTHDATE']), 
                     (encounters_df, ['START', 'STOP']), 
                     (conditions_df, ['START']), 
                     (meds_df, ['START'])]:
        for col in cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col]).dt.tz_localize(None)

    # --- 3. Identify Index Admission and Engineer Target Variable ---
    encounters_df = encounters_df.sort_values(by=['PATIENT', 'START'])
    inpatient_df = encounters_df[encounters_df['ENCOUNTERCLASS'] == 'inpatient'].copy()
    
    if inpatient_df.empty:
        print(f"Warning: No inpatient encounters found for {data_path.name}. Returning empty DataFrames.")
        return pd.DataFrame(), pd.Series()

    # The "index admission" is the first inpatient stay for each patient
    index_admissions = inpatient_df.groupby('PATIENT').first().reset_index()

    # Find any subsequent encounter within 30 days of the index admission's discharge
    merged_encounters = pd.merge(
        encounters_df,
        index_admissions[['PATIENT', 'STOP']].rename(columns={'STOP': 'INDEX_DISCHARGE'}),
        on='PATIENT'
    )
    
    readmission_mask = (
        (merged_encounters['START'] > merged_encounters['INDEX_DISCHARGE']) &
        (merged_encounters['START'] <= merged_encounters['INDEX_DISCHARGE'] + pd.Timedelta(days=30))
    )
    readmitted_patients = merged_encounters[readmission_mask]['PATIENT'].unique()
    index_admissions['readmitted_30d'] = index_admissions['PATIENT'].isin(readmitted_patients).astype(int)

    # --- 4. Feature Engineering ---
    # Merge demographics
    model_df = pd.merge(
        index_admissions,
        patients_df[['Id', 'BIRTHDATE', 'GENDER', 'RACE', 'ETHNICITY', 'ZIP']],
        left_on='PATIENT',
        right_on='Id',
        how='left'
    )
    
    # Age at admission
    model_df['PATIENT_AGE'] = ((model_df['START'] - model_df['BIRTHDATE']).dt.days / 365.25).apply(lambda x: max(0, x)).astype(int)
    
    # Length of stay for the index admission
    model_df['length_of_stay'] = (model_df['STOP'] - model_df['START']).dt.days

    # Utilization features (events in 180 days *before* index admission)
    util_features = []
    for _, row in model_df.iterrows():
        patient_id = row['PATIENT']
        admission_date = row['START']
        
        # Filter events before the admission
        prior_encounters = encounters_df[(encounters_df['PATIENT'] == patient_id) & (encounters_df['START'] < admission_date) & (encounters_df['START'] >= admission_date - pd.Timedelta(days=180))]
        prior_conditions = conditions_df[(conditions_df['PATIENT'] == patient_id) & (conditions_df['START'] < admission_date) & (conditions_df['START'] >= admission_date - pd.Timedelta(days=365))]
        prior_meds = meds_df[(meds_df['PATIENT'] == patient_id) & (meds_df['START'] < admission_date) & (meds_df['START'] >= admission_date - pd.Timedelta(days=180))]
        
        util_features.append({
            'PATIENT': patient_id,
            'util_180d_encounters': len(prior_encounters),
            'util_180d_inpatient': len(prior_encounters[prior_encounters['ENCOUNTERCLASS'] == 'inpatient']),
            'num_prior_conditions': len(prior_conditions),
            'num_prior_meds': prior_meds['CODE'].nunique()
        })
    
    model_df = pd.merge(model_df, pd.DataFrame(util_features), on='PATIENT', how='left')

    # --- 5. Final Feature Selection and Encoding ---
    feature_cols = [
        'PATIENT_AGE', 'GENDER', 'RACE', 'ETHNICITY', 'ZIP', 'length_of_stay',
        'util_180d_encounters', 'util_180d_inpatient', 'num_prior_conditions', 'num_prior_meds'
    ]
    target_col = 'readmitted_30d'
    
    final_df = model_df[feature_cols + [target_col]].copy().fillna(0)

    # One-Hot Encode categorical features
    categorical_cols = ['GENDER', 'RACE', 'ETHNICITY', 'ZIP']
    final_df = pd.get_dummies(final_df, columns=categorical_cols, drop_first=True)

    # --- 6. Create Final X and y DataFrames ---
    X = final_df.drop(columns=target_col)
    y = final_df[target_col]
    
    return X, y


In [31]:
# --- Process Data for All Hospitals ---
all_data = {}
for hospital_id in ['A', 'B', 'C']:
    print(f"--- Building Cohort for Hospital {hospital_id} ---")
    data_path = Path(f"/Users/faezehhosseini/phd_federated_learning_project/data/hospital_{hospital_id}/csv")
    X, y = build_readmission_cohort(data_path)
    if not X.empty:
        print(f"Hospital {hospital_id}: {X.shape[0]} patients, {y.sum()} readmissions")
        all_data[hospital_id] = (X, y)

# --- Save Processed Data to Disk ---
PROCESSED_DATA_PATH = Path("processed_data")
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Align columns across all datasets to ensure consistency
all_X_dfs = [data[0] for data in all_data.values()]
if all_X_dfs:
    # Get all unique columns from all dataframes
    all_cols = pd.concat(all_X_dfs, axis=0, sort=False).columns
    
    for hospital_id, (X, y) in all_data.items():
        # Reindex each X to have all columns, filling missing with 0
        X_aligned = X.reindex(columns=all_cols, fill_value=0)
        
        # Save the aligned data
        X_aligned.to_csv(PROCESSED_DATA_PATH / f"X_{hospital_id}.csv", index=False)
        y.to_csv(PROCESSED_DATA_PATH / f"y_{hospital_id}.csv", index=False)
        print(f"Saved processed data for Hospital {hospital_id}")

print("\n✅ Preprocessing and saving complete.")


--- Building Cohort for Hospital A ---
Hospital A: 341 patients, 84 readmissions
--- Building Cohort for Hospital B ---
Hospital B: 334 patients, 78 readmissions
--- Building Cohort for Hospital C ---
Hospital C: 462 patients, 101 readmissions
Saved processed data for Hospital A
Saved processed data for Hospital B
Saved processed data for Hospital C

✅ Preprocessing and saving complete.
